In [1]:
import duckdb
import polars as pl
import pendulum
from pathlib import Path

data_dir = Path("../data/lastfm/listening")
db = duckdb.read_json(data_dir/"*.jsonl")
df = db.pl()

df = df.with_columns(
    pl.from_epoch(
        pl.col("date_played_unix"), time_unit="s"
    ).alias("track_played_utc")
)

print("Raw data:", df.shape)

df = df.unique("date_played_unix") # Not so exact with which duplicate gets removed
print("Duplicate timestamps removed:", df.unique("date_played_unix").shape)
df = df.with_columns(
    artist_name = pl.col("artist_name").str.to_lowercase(),
    track_name = pl.col("track_name").str.to_lowercase(),
    album_name = pl.col("album_name").str.to_lowercase()
)

Raw data: (110513, 8)
Duplicate timestamps removed: (108066, 8)


In [2]:
df.sort("track_played_utc", descending=True).head()

artist_name,artist_mbid,album_name,album_mbid,track_name,track_mbid,date_played_unix,track_played_utc
str,str,str,str,str,str,i64,datetime[μs]
"""andrew aged""","""8a0c3a4f-8115-4261-9f42-30b660…","""crown""","""3556324c-7380-40b9-b6e6-2f1722…","""banner""","""""",1774463432,2026-03-25 18:30:32
"""lloyd""","""d75b3154-fb27-479b-b40b-b1b718…","""street love""","""55f161cd-e1fb-46f0-9b2f-a26578…","""get it shawty""","""04f476ec-18e6-3003-8b0b-5d3d4c…",1774462669,2026-03-25 18:17:49
"""yeat""","""9e6a6a2f-f696-4cc0-87f1-e4f712…","""alivë""","""""","""diëd b4""","""""",1774462486,2026-03-25 18:14:46
"""maya ricci""","""""","""syko""","""""","""syko""","""""",1774462362,2026-03-25 18:12:42
"""brakence""","""""","""punk2""","""bad4bc48-356b-45fe-b8ad-d2b897…","""ginger tea""","""c0645055-8b90-4e90-8222-f09b80…",1774462214,2026-03-25 18:10:14


## Top artists by play count

In [3]:
# Top artists of all-time
(df
 .group_by("artist_name")
 .agg(total_plays = pl.len())
 .sort("total_plays", descending=True)
).head(20)

artist_name,total_plays
str,u32
"""bladee""",4356
"""teen suicide""",1586
"""yung lean""",1515
"""charli xcx""",1157
"""yeat""",1047
…,…
"""julia brown""",704
"""aphex twin""",679
"""cocteau twins""",607


In [4]:
# Top artist for a year
(df
 .filter(
    pl.col("track_played_utc") >= pendulum.datetime(2024, 1, 1).naive(),
    pl.col("track_played_utc") < pendulum.datetime(2025, 1, 1).naive(),
 )
 .group_by("artist_name")
 .agg(total_scrobbles = pl.len())
 .sort("total_scrobbles", descending=True)
).head(20)

def top_artists_yearly(df, year:int=2025):
   df = (df
      .filter(
         pl.col("track_played_utc") >= pendulum.datetime(year, 1, 1).naive(),
         pl.col("track_played_utc") < pendulum.datetime(year+1, 1, 1).naive(),
      )
      .group_by("artist_name")
      .agg(total_scrobbles = pl.len())
      .sort("total_scrobbles", descending=True)
   ) 
   return df

### Artists and their played tracks

In [5]:
df_track_count = (df
 .filter(
    pl.col("track_played_utc").dt.year() == 2025,
 )
 .group_by(pl.col( "artist_name" ), pl.col("track_name"))
 .agg(total_scrobbles = pl.len())
 .sort("total_scrobbles", descending=True)
)

df_artists = (df
 .filter(
    pl.col("track_played_utc").dt.year() == 2025,
 )
 .group_by(pl.col( "artist_name" ))
 .agg(artist_scrobbles = pl.len())
 .sort("artist_scrobbles", descending=True)
 .head(10)
)
print(df_artists.select("artist_name"))
df_track_count.join(df_artists, on="artist_name", how="semi").select(pl.col("artist_name").unique(maintain_order=True))

shape: (10, 1)
┌──────────────┐
│ artist_name  │
│ ---          │
│ str          │
╞══════════════╡
│ bladee       │
│ dean blunt   │
│ bassvictim   │
│ astrid sonne │
│ vegyn        │
│ smerz        │
│ c418         │
│ dijon        │
│ ulla         │
│ mk.gee       │
└──────────────┘


artist_name
str
"""dean blunt"""
"""bassvictim"""
"""astrid sonne"""
"""bladee"""
"""dijon"""
"""smerz"""
"""ulla"""
"""vegyn"""
"""c418"""


In [6]:
import plotly.express as px
px.bar(
q1.join(q2, on="artist_name", how="semi")
, x="total_scrobbles", y="artist_name", hover_data=["total_scrobbles", "artist_name", "track_name"])

NameError: name 'q1' is not defined

In [ ]:
q = (df
 .filter(pl.col("track_played_utc").dt.year() == 2025)
 .group_by("artist_name", "track_name")
 .agg(total_scrobbles=pl.len())
 .with_columns(
     artist_scrobbles=pl.col("total_scrobbles").sum().over("artist_name")
 )
 .sort("artist_scrobbles", descending=True)
)

top_10 = q.select("artist_name").unique(maintain_order=True).head(10)
# Filter combined data to only those artists
q.join(top_10, on="artist_name", how="semi")

artist_name,track_name,total_scrobbles,artist_scrobbles
str,str,u32,u32
"""bladee""","""end of the road boyz""",2,459
"""bladee""","""wonderland""",1,459
"""bladee""","""yung sherman (feat yung sherma…",7,459
"""bladee""","""missing person""",3,459
"""bladee""","""so cold interlude""",2,459
…,…,…,…
"""mk.gee""","""rockman""",16,223
"""mk.gee""","""are you looking up""",25,223
"""mk.gee""","""candy""",13,223


## Artist loyalty score — what % of your listening is your top 10?


In [ ]:
top_artists = (
    df
    .group_by("artist_name")
    .agg(total_scrobbles = pl.len())
    .sort("total_scrobbles", descending=True)
)
top_10_scrobbles = (
    top_artists
    .select(
        pl.col("total_scrobbles").limit(10).sum().alias("top_10_total_scrobbles"),
        pl.col("total_scrobbles").sum().alias("total_scrobbles")
    )
    .with_columns(
        (pl.col("top_10_total_scrobbles") / pl.col("total_scrobbles"))
        .alias("artist_loyalty_score")
    )
)
top_10_scrobbles

top_10_total_scrobbles,total_scrobbles,artist_loyalty_score
u32,u32,f64
14558,108066,0.134714


## New artist discoveries over time (first time you played an artist)

In [ ]:
q = (
    df
    .select(pl.col("artist_name"), pl.col("track_played_utc"))
    
    .group_by(pl.col("artist_name"))
    .agg(
        first_played = pl.col("track_played_utc").min(),
        last_played = pl.col("track_played_utc").max(),
        total_scrobbles = pl.len(),
    )
    .sort("total_scrobbles", descending=True)
    .limit(200)
    .filter(pl.col("first_played") > pendulum.datetime(2022, 1, 1).naive())
)

q

artist_name,first_played,last_played,total_scrobbles
str,datetime[μs],datetime[μs],u32
"""snow strippers""",2024-01-09 12:33:21,2026-03-02 13:05:03,1039
"""destroy lonely""",2022-04-22 14:45:46,2025-12-30 14:50:08,732
"""ken carson""",2022-07-05 19:07:16,2026-01-30 18:59:18,550
"""bassvictim""",2024-07-15 11:23:39,2026-03-23 09:28:20,506
"""dean blunt""",2023-06-30 11:04:35,2026-03-24 09:43:09,473
…,…,…,…
"""jasper tygner""",2023-06-29 17:36:28,2026-03-14 15:02:26,119
"""emile mosseri""",2025-01-07 07:39:45,2026-03-24 21:13:21,118
"""jai paul""",2023-01-25 10:36:09,2025-07-05 16:33:16,113


In [ ]:
for idx, a in enumerate(q.iter_rows(), 1):
    print(idx, a)

1 ('snow strippers', datetime.datetime(2024, 1, 9, 12, 33, 21), datetime.datetime(2026, 3, 2, 13, 5, 3), 1039)
2 ('destroy lonely', datetime.datetime(2022, 4, 22, 14, 45, 46), datetime.datetime(2025, 12, 30, 14, 50, 8), 732)
3 ('ken carson', datetime.datetime(2022, 7, 5, 19, 7, 16), datetime.datetime(2026, 1, 30, 18, 59, 18), 550)
4 ('bassvictim', datetime.datetime(2024, 7, 15, 11, 23, 39), datetime.datetime(2026, 3, 23, 9, 28, 20), 506)
5 ('dean blunt', datetime.datetime(2023, 6, 30, 11, 4, 35), datetime.datetime(2026, 3, 24, 9, 43, 9), 473)
6 ('caroline polachek', datetime.datetime(2023, 5, 1, 13, 45, 8), datetime.datetime(2026, 3, 25, 13, 34, 23), 439)
7 ('smerz', datetime.datetime(2023, 12, 26, 6, 22, 58), datetime.datetime(2026, 3, 23, 9, 25, 30), 403)
8 ('mall grab', datetime.datetime(2022, 4, 13, 11, 29, 14), datetime.datetime(2026, 3, 20, 18, 32, 38), 401)
9 ('1tbsp', datetime.datetime(2022, 10, 19, 9, 48, 54), datetime.datetime(2026, 3, 25, 13, 49), 382)
10 ('astrid sonne', da

## Artist variety index – how diverse is your listening?

In [ ]:
top_artists_25 = top_artists_yearly(df, 2025)
top_artists_25.with_columns(
    pct_listened = pl.col("total_scrobbles") / pl.col("total_scrobbles").sum()
)
# total_scrobbles_25 = top_artists_25.select(pl.col("total_plays").sum()).item()

artist_name,total_scrobbles,pct_listened
str,u32,f64
"""bladee""",459,0.022894
"""dean blunt""",411,0.0205
"""bassvictim""",369,0.018405
"""astrid sonne""",301,0.015013
"""vegyn""",301,0.015013
…,…,…
"""allem iversom""",1,0.00005
"""diamond*""",1,0.00005
"""melt-banana""",1,0.00005


https://www.reddit.com/r/lastfm/comments/x9eq6s/what_is_your_artist_diversity_ratio/

$$
\text{Artist Diversity Ratio} = \frac{\text{total number of scrobbles}}{\text{number of unique artists}}
$$

The lower, the better.

In [ ]:
top_artists_25.select(
    ( pl.col("total_scrobbles").sum() / pl.len() ).alias("artist_diversity_ratio")
)

artist_diversity_ratio
f64
12.909852


## Artists you've "abandoned" vs. artists you keep returning to

Idea to get "relevant" artists.
1. Calculate mean and standard deviations 
2. Cut-off left tail, only including observations equal to 2 standard deviations or greater

In [ ]:
(
    df
    .select(pl.col("artist_name"), pl.col("track_played_utc"))
    
    .group_by(pl.col("artist_name"))
    .agg(
        penultimate_played = pl.col("track_played_utc").top_k(2).last(),
        last_played = pl.col("track_played_utc").max(),
        total_scrobbles = pl.len(),
    )
    .with_columns(
        tdelta = (pl.col("last_played") - pl.col("penultimate_played")).dt.total_days()
        )
    .sort("tdelta", descending=True)
)

artist_name,penultimate_played,last_played,total_scrobbles,tdelta
str,datetime[μs],datetime[μs],u32,i64
"""plain white t's""",2021-10-16 09:14:14,2026-01-06 17:08:55,2,1543
"""tek genesis""",2022-04-02 15:18:58,2026-01-14 14:46:07,4,1382
"""jump man 93""",2021-10-15 10:39:43,2025-05-08 16:07:44,37,1301
"""lilly wood & the prick""",2022-05-02 08:59:47,2025-10-16 08:30:52,3,1262
"""swim rest""",2021-06-28 19:38:10,2024-11-14 20:49:38,4,1235
…,…,…,…,…
"""slaughter beach, dog""",2021-10-07 17:28:30,2021-10-07 17:28:30,1,0
"""dna""",2025-04-19 08:22:04,2025-04-19 08:22:04,1,0
"""jay p""",2025-11-14 15:03:43,2025-11-14 15:03:43,1,0


In [ ]:
(
    df
    .select(pl.col("artist_name"), pl.col("track_played_utc"))
    
    .group_by(pl.col("artist_name"))
    .agg(
        penultimate_played = pl.col("track_played_utc").top_k(2).last(),
        last_played = pl.col("track_played_utc").max(),
        total_scrobbles = pl.len(),
    )
    # .filter(pl.col("total_scrobbles").quantile())
    .with_columns(
        tdelta = (pl.col("last_played") - pl.col("penultimate_played")).dt.total_days()
        )
    .sort("tdelta", descending=True)
)

artist_name,penultimate_played,last_played,total_scrobbles,tdelta
str,datetime[μs],datetime[μs],u32,i64
"""plain white t's""",2021-10-16 09:14:14,2026-01-06 17:08:55,2,1543
"""tek genesis""",2022-04-02 15:18:58,2026-01-14 14:46:07,4,1382
"""jump man 93""",2021-10-15 10:39:43,2025-05-08 16:07:44,37,1301
"""lilly wood & the prick""",2022-05-02 08:59:47,2025-10-16 08:30:52,3,1262
"""swim rest""",2021-06-28 19:38:10,2024-11-14 20:49:38,4,1235
…,…,…,…,…
"""chris rea""",2021-12-21 19:25:14,2021-12-21 19:25:14,1,0
"""vieux farka touré""",2024-05-14 10:54:55,2024-05-14 10:54:55,1,0
"""ibukun sunday""",2025-02-17 13:52:59,2025-02-17 13:52:59,1,0


In [ ]:
# Sorted by 'scrobbles'
(
    df
    .select(pl.col("artist_name"), pl.col("track_played_utc"))
    
    .group_by(pl.col("artist_name"))
    .agg(
        penultimate_played = pl.col("track_played_utc").top_k(2).last(),
        last_played = pl.col("track_played_utc").max(),
        total_scrobbles = pl.len(),
    )
    .filter(pl.col("total_scrobbles") >= pl.col("total_scrobbles").quantile(0.60)) 
    .with_columns(
        tdelta = (pl.col("last_played") - pl.col("penultimate_played"))
        )
    .sort("total_scrobbles", descending=True)
)

artist_name,penultimate_played,last_played,total_scrobbles,tdelta
str,datetime[μs],datetime[μs],u32,duration[μs]
"""bladee""",2026-03-24 13:58:30,2026-03-25 14:31:58,4356,1d 33m 28s
"""teen suicide""",2026-03-20 07:44:00,2026-03-24 09:45:10,1586,4d 2h 1m 10s
"""yung lean""",2026-03-23 15:09:26,2026-03-23 15:30:37,1515,21m 11s
"""charli xcx""",2026-02-15 11:31:35,2026-03-01 12:38:41,1157,14d 1h 7m 6s
"""yeat""",2025-09-03 10:22:57,2026-03-25 18:14:46,1047,203d 7h 51m 49s
…,…,…,…,…
"""shiro sagisu""",2021-12-03 15:17:18,2022-02-01 22:20:16,5,60d 7h 2m 58s
"""plan8""",2026-01-11 09:24:04,2026-02-12 08:44:33,5,31d 23h 20m 29s
"""the symposium""",2024-11-01 20:45:32,2024-11-27 08:55:33,5,25d 12h 10m 1s


In [ ]:
# Sorted by 'tdelta'
(
    df
    .select(pl.col("artist_name"), pl.col("track_played_utc"))
    
    .group_by(pl.col("artist_name"))
    .agg(
        penultimate_played = pl.col("track_played_utc").top_k(2).last(),
        last_played = pl.col("track_played_utc").max(),
        total_scrobbles = pl.len(),
    )
    .filter(pl.col("total_scrobbles") >= pl.col("total_scrobbles").quantile(0.60)) 
    .with_columns(
        tdelta = (pl.col("last_played") - pl.col("penultimate_played"))
        )
    .sort("tdelta", descending=False)
).head(30)

artist_name,penultimate_played,last_played,total_scrobbles,tdelta
str,datetime[μs],datetime[μs],u32,duration[μs]
"""julie""",2025-08-10 17:09:11,2025-08-10 17:09:11,25,0µs
"""ripsquad archive""",2026-01-15 07:20:43,2026-01-15 07:20:43,24,0µs
"""dapurr""",2026-01-30 16:53:16,2026-01-30 16:53:16,31,0µs
"""felicity j lord""",2025-09-29 20:03:18,2025-09-29 20:03:18,21,0µs
"""sammy virji""",2026-01-18 14:37:53,2026-01-18 14:37:53,34,0µs
…,…,…,…,…
"""michael small""",2025-01-15 13:13:21,2025-01-15 13:14:46,8,1m 25s
"""imagine drowning""",2026-03-23 08:45:19,2026-03-23 08:46:45,287,1m 26s
"""solange""",2025-02-28 17:48:21,2025-02-28 17:49:47,12,1m 26s


In [ ]:
# Sorted by tdelta
df_scrobbles = (
    df
    .group_by(pl.col("artist_name"))
    .agg(
        total_scrobbles = pl.len(),
    )
    .filter(pl.col("total_scrobbles") >= 5) 
).sort("total_scrobbles", descending=False) # Removing any artist I've listened to less than 5 times

df_tdelta = ( 
    df.join(df_scrobbles, on="artist_name", how="right")
    .select(pl.col("artist_name"), pl.col("track_played_utc"), pl.col("total_scrobbles"))
    .group_by(pl.col("artist_name"))
    .agg(
        total_scrobbles = pl.col("total_scrobbles").first(),
        penultimate_played = pl.col("track_played_utc").sort(descending=True).get(1), # By ensuring that we have >1 entry for every artist, .get(1) will not get an index error
        last_played = pl.col("track_played_utc").max())
    .with_columns(
        tdelta = (pl.col("last_played") - pl.col("penultimate_played"))
        )
    .sort("tdelta", descending=False)
)
df_tdelta

ComputeError: get index is out of bounds

This error occurred in the following expression:
	col("track_played_utc").sort(desc).get(1)


In [ ]:
# Sorted by total_scrobbles 
df_scrobbles = (
    df
    .group_by(pl.col("artist_name"))
    .agg(
        total_scrobbles = pl.len(),
    )
    .filter(pl.col("total_scrobbles") >= 5) 
).sort("total_scrobbles", descending=False) # Removing any artist I've listened to less than 5 times

df_tdelta = ( 
    df.join(df_scrobbles, on="artist_name", how="right")
    .select(pl.col("artist_name"), pl.col("track_played_utc"), pl.col("total_scrobbles"))
    .group_by(pl.col("artist_name"))
    .agg(
        total_scrobbles = pl.col("total_scrobbles").first(),
        penultimate_played = pl.col("track_played_utc").sort(descending=True).get(1), # By ensuring that we have >1 entry for every artist, .get(1) will not get an index error
        last_played = pl.col("track_played_utc").max())
    .with_columns(
        tdelta = (pl.col("last_played") - pl.col("penultimate_played"))
        )
    .sort("total_scrobbles", descending=True)
)
df_tdelta

ComputeError: get index is out of bounds

This error occurred in the following expression:
	col("track_played_utc").sort(desc).get(1)


In [ ]:
# Tdelta, but it time since I lasted played a song by the artist.

( 
    df.join(df_scrobbles, on="artist_name", how="right")
    .select(pl.col("artist_name"), pl.col("track_played_utc"), pl.col("total_scrobbles"))
    .group_by(pl.col("artist_name"))
    .agg(
        total_scrobbles = pl.col("total_scrobbles").first(),
        last_played = pl.col("track_played_utc").max())
    .with_columns(
        # tdelta = (pendulum.datetime(2026, 3, 27).naive()-pl.col("last_played"))
        tdelta = (pendulum.today().naive()-pl.col("last_played"))
        )
    .sort("total_scrobbles", descending=True)
)

artist_name,total_scrobbles,last_played,tdelta
str,u32,datetime[μs],duration[μs]
"""bladee""",4356,2026-03-25 14:31:58,9d 9h 28m 2s
"""teen suicide""",1586,2026-03-24 09:45:10,10d 14h 14m 50s
"""yung lean""",1515,2026-03-23 15:30:37,11d 8h 29m 23s
"""charli xcx""",1157,2026-03-01 12:38:41,33d 11h 21m 19s
"""yeat""",1047,2026-03-25 18:14:46,9d 5h 45m 14s
…,…,…,…
"""modest mouse""",5,2025-12-31 16:10:25,93d 7h 49m 35s
"""exum""",5,2025-10-26 15:23:25,159d 8h 36m 35s
"""shiro sagisu""",5,2022-02-01 22:20:16,1522d 1h 39m 44s
